# GEDI Canopy Height Pipeline for Route 29 North

This notebook processes NASA GEDI Level 2A HDF5 files stored in S3 through four sequential phases:

| Phase | Description |
|---|---|
| **Phase 1** | High-performance batch extraction of GEDI shots from S3 |
| **Phase 1a** | Year parsing from GEDI filenames |
| **Phase 1b** | Harmonization to the SMAP 9 km grid |
| **Phase 1c** | Spatial join to assign shots to study-area jurisdictions |

**Source:** `s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/`  
**Final summary output:** `s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary-route29north.csv`
**Final detailed output:** `s3://central-virginia-tree-canopy-project/gedi-county-detailed/virginia_gedi_county_detailed-route29north.csv`

## Install Dependencies

In [1]:
import subprocess, sys

# Pin the entire boto stack atomically — all four packages must be compatible
boto_stack = [
    "aiobotocore==3.7.0",
    "botocore==1.43.0",
    "boto3==1.43.0",
    "s3fs==2026.6.0",
]

# Install boto stack with --no-deps to prevent pip from overriding the pins
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "--force-reinstall"] + boto_stack)

# Install all other dependencies normally
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "aioitertools", "h5py", "geopandas", "xarray", "netCDF4", "pyarrow"])

print("All dependencies installed.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.45.39 requires botocore==1.43.39, but you have botocore 1.43.0 which is incompatible.
awscli 1.45.39 requires s3transfer<0.20.0,>=0.19.0, but you have s3transfer 0.17.1 which is incompatible.
sagemaker 2.257.3 requires attrs<26,>=24, but you have attrs 26.1.0 which is incompatible.


All dependencies installed.


## Imports

In [2]:
import os
import time
import io
import re
import urllib.request

from concurrent.futures import ThreadPoolExecutor, as_completed

import statistics

import boto3
import s3fs
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

print("Imports successful.")

Imports successful.


## Configuration
Edit this cell to change S3 paths, bounding box, grid resolution, or target jurisdictions.

In [3]:
# ── S3 settings ───────────────────────────────────────────────────────────────
S3_BUCKET             = "central-virginia-tree-canopy-project"
S3_PREFIX             = "GEDI/GEDI02_A/002/"     
GEDI_COUNTY_S3_PREFIX = "gedi-county-summary/"
GEDI_DETAILED_COUNTY_S3_PREFIX = "gedi-county-detailed/"
GEDI_INFO_KEY = "data/gedi-folder/route29-north/gedi02_A_forRoute29North.txt"

# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR        = "/home/ec2-user/SageMaker/gedi_output"
OUTPUT_PARQUET    = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_multiyear-route29north.parquet")
OUTPUT_NETCDF     = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_grid-route29north.nc")
OUTPUT_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_summary-route29north.csv")
OUTPUT_DETAILED_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_detailed-route29north.csv")
OUTPUT_DETAILED_CANOPY_HEIGHT_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_height-route29north.csv")
OUTPUT_COUNTY_CANOPY_HEIGHT_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_canopy_height-route29north.csv")

# ── Spatial bounds (Route 29 North Corridor (Places29 / Airport / Hollymead Sector) jurisdiction study area) ───────────────────────
MIN_LON, MIN_LAT, MAX_LON, MAX_LAT = -78.4750, 38.0650, -78.3950, 38.2320

# ── SMAP grid resolution (~9 km) ─────────────────────────────────────────────
GRID_RES = 0.081

# ── Target jurisdictions (Name, State FIPS, County/Place FIPS, Type) ─────────
TARGET_JURISDICTIONS = [
    ("Albemarle",       "51", "003",   "county"),
    ("Augusta",         "51", "015",   "county"),
    ("Buckingham",      "51", "029",   "county"),
    ("Charlottesville", "51", "14968", "place"),
    ("Fluvanna",        "51", "065",   "county"),
    ("Greene",          "51", "079",   "county"),
    ("Louisa",          "51", "109",   "county"),
    ("Nelson",          "51", "125",   "county"),
    ("Rockingham",      "51", "165",   "county"),
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  GEDI source  : s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  S3 output    : s3://{S3_BUCKET}/{GEDI_COUNTY_S3_PREFIX}")

Configuration loaded.
  GEDI source  : s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/
  Output dir   : /home/ec2-user/SageMaker/gedi_output
  S3 output    : s3://central-virginia-tree-canopy-project/gedi-county-summary/


## Helper Functions

In [4]:
def parse_year_from_filename(filename: str):
    """Extract the year from a standard GEDI filename (e.g., GEDI02_A_2022143...)."""
    year_match = re.search(r'GEDI02_A_(\d{4})', filename)
    if year_match:
        return int(year_match.group(1))
    return None


def fetch_boundary(name: str, state_fips: str, geo_id: str, geo_type: str) -> gpd.GeoDataFrame:
    """Fetch boundary GeoJSON directly from the US Census TIGERweb API."""
    if geo_type == "place":
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "Places_CouSub_ConCity_SubMCD/MapServer/4/query"
            f"?where=STATE='{state_fips}'+AND+PLACE='{geo_id}'"
            "&outFields=NAME,STATE,PLACE&f=geojson&outSR=4326"
        )
    else:
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "State_County/MapServer/11/query"
            f"?where=STATE='{state_fips}'+AND+COUNTY='{geo_id}'"
            "&outFields=NAME,STATE,COUNTY&f=geojson&outSR=4326"
        )
    print(f"  Fetching boundary for {name}...")
    with urllib.request.urlopen(url, timeout=20) as r:
        gdf = gpd.read_file(r)
    if gdf.empty:
        raise ValueError(f"No boundary found for {name} (FIPS {state_fips}/{geo_id})")
    gdf = gdf.set_crs("EPSG:4326")
    gdf['jurisdiction'] = name
    return gdf

def extract_h5_filename(url: str) -> str:
    """
    Extract just the .h5 file name (basename) from a full granule URL.
 
    Parameters
    ----------
    url : str
        Full URL to a .h5 granule, e.g.
        https://.../GEDI02_A_2025189013841_..._V002/GEDI02_A_2025189013841_..._V002.h5
 
    Returns
    -------
    str
        Just the filename, e.g. GEDI02_A_2025189013841_..._V002.h5
    """
    path = urlparse(url).path
    return posixpath.basename(path)
 
 
def read_url_list_from_s3(bucket: str, key: str, profile: str = None) -> list[str]:
    """
    Fetch a text object from S3 and return the .h5 filename from each non-empty line.
 
    Parameters
    ----------
    bucket : str
        S3 bucket name.
    key : str
        S3 object key (path within the bucket).
    profile : str, optional
        Named AWS CLI profile to use. If None, uses the default credential chain.
 
    Returns
    -------
    list[str]
        List of .h5 file names, one per non-blank line in the source file.
    """
    session = boto3.Session(profile_name=profile) if profile else boto3.Session()
    s3 = session.client("s3")
 
    try:
        #log.info("Fetching s3://%s/%s ...", bucket, key)
        print(f"Fetching s3://{bucket}/{key} ...")
        response = s3.get_object(Bucket=bucket, Key=key)
        body = response["Body"].read().decode("utf-8")
    except NoCredentialsError:
        #log.error("No AWS credentials found. Configure ~/.aws/credentials, "
        #           "set AWS_PROFILE, or pass --profile.")
        print("No AWS credentials found. Configure ~/.aws/credentials, set AWS_PROFILE, or pass --profile.")

        raise
    except ClientError as e:
        error_code = e.response.get("Error", {}).get("Code", "Unknown")
        if error_code == "NoSuchKey":
            #log.error("Object not found: s3://%s/%s", bucket, key)
            print(f"Object not found: s3://{bucket}/{key}")
        elif error_code in ("AccessDenied", "403"):
            print("TEST")
        else:
            #log.error("S3 error (%s) reading s3://%s/%s: %s", error_code, bucket, key, e)
            print(f"S3 error ({error_code}) reading s3://{bucket}/{key}: {e}")
        raise
 
    lines = [line.strip() for line in body.splitlines() if line.strip()]
    filenames = [extract_h5_filename(line) for line in lines]
    #log.info("Read %d line(s) from s3://%s/%s and extracted %d .h5 filename(s)",
    #          len(lines), bucket, key, len(filenames))
    print(f"Read {len(lines)} line(s) from s3://{bucket}/{key} and extracted {len(filenames)} .h5 filename(s)")
    return filenames

# This update works

def process_one_file(key):
    """
    Download one GEDI .h5 granule from S3 into memory (single bulk transfer,
    not lazy range-requests -- HDF5's internal metadata reads make lazy
    access over a network filesystem far slower due to per-request latency),
    then apply spatial + quality filtering. Returns a list of per-beam
    DataFrames (possibly empty).
    """
    file_name = os.path.basename(key)
    results = []

    year = parse_year_from_filename(file_name)
    if not year:
        print(f"Warning: Could not parse year from {file_name}. Skipping.")
        return results

    try:
        s3_path = f"s3://{S3_BUCKET}/{key}"
        with fs.open(s3_path, "rb") as f:
            raw_bytes = f.read()

        with h5py.File(io.BytesIO(raw_bytes), "r") as hf:
            beams = [k for k in hf.keys() if k.startswith("BEAM")]
            for beam in beams:
                if "lat_lowestmode" not in hf[beam]:
                    continue

                lats = hf[f"{beam}/lat_lowestmode"][:]
                lons = hf[f"{beam}/lon_lowestmode"][:]
                spatial_mask = (
                    (lons >= MIN_LON) & (lons <= MAX_LON) &
                    (lats >= MIN_LAT) & (lats <= MAX_LAT)
                )
                if not np.any(spatial_mask):
                    continue

                quality     = hf[f"{beam}/quality_flag"][:][spatial_mask]
                sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
                rh98        = hf[f"{beam}/rh"][:, 98][spatial_mask]

                beam_df = pd.DataFrame({
                    "longitude":    lons[spatial_mask],
                    "latitude":     lats[spatial_mask],
                    "quality_flag": quality,
                    "sensitivity":  sensitivity,
                    "rh98":         rh98,
                    "year":         year,
                    "file_source":  file_name,
                    "beam":         beam
                })

                valid_df = beam_df[
                    (beam_df["quality_flag"] == 1) &
                    (beam_df["sensitivity"]  > 0.9) &
                    (beam_df["rh98"]         > 0)   &
                    (beam_df["rh98"]         < 100)
                ]
                if not valid_df.empty:
                    results.append(valid_df)

    except Exception as e:
        print(f"Error reading {file_name}: {e}")

    return results

print("Helper functions defined.")

Helper functions defined.


In [5]:
from urllib.parse import urlparse
import posixpath

from botocore.exceptions import ClientError

s3_client = boto3.client('s3')

county_specific_gedi_files = []
missing_files = []

filenames = read_url_list_from_s3(S3_BUCKET, GEDI_INFO_KEY, profile=None)

print(f"\n{len(filenames)} GEDI .h5 file name(s):\n")

for name in filenames:
    county_specific_gedi_files.append(name)
    #Does the filename exist in our data repository?
    #print("Checking file existence in S3...")
    # Construct the full S3 key path for the file
    s3_key = f"{S3_PREFIX}{name}"
    try:
        # head_object returns metadata if the file exists
        s3_client.head_object(Bucket=S3_BUCKET, Key=s3_key)
        #existing_files.append(filename)
        #print(f"[EXISTS] s3://{S3_BUCKET}/{s3_key}")
        
    except ClientError as e:
        # An error response code of 404 means the file does not exist
        if e.response['Error']['Code'] == "404":
            missing_files.append(name)
            print(f"[MISSING] s3://{S3_BUCKET}/{s3_key}")
        else:
            # Handle other AWS permissions or network errors (e.g., 403 Forbidden)
            print(f"[ERROR] Could not check {name}: {e.response['Error']['Message']}")
    
    #print(name)

print(f"Found {len(county_specific_gedi_files)} GEDI County Specific HDF5 files.")
if county_specific_gedi_files:
    print(f"  First : {county_specific_gedi_files[0]}")
    print(f"  Last  : {county_specific_gedi_files[-1]}")


Fetching s3://central-virginia-tree-canopy-project/data/gedi-folder/route29-north/gedi02_A_forRoute29North.txt ...
Read 52 line(s) from s3://central-virginia-tree-canopy-project/data/gedi-folder/route29-north/gedi02_A_forRoute29North.txt and extracted 52 .h5 filename(s)

52 GEDI .h5 file name(s):

Found 52 GEDI County Specific HDF5 files.
  First : GEDI02_A_2025173142104_O36969_03_T05926_02_004_02_V002.h5
  Last  : GEDI02_A_2019141235737_O02487_02_T00138_02_003_01_V002.h5


## Phase 1 & 1a: Discover GEDI Files in S3

In [6]:
print(f"Scanning s3://{S3_BUCKET}/{S3_PREFIX} for GEDI HDF5 files...Only use county specific files.")

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

h5_keys = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".h5"):
            #Check for a match in the county_specific_gedi_files array
            target_filename = posixpath.basename(obj["Key"])
            if target_filename in county_specific_gedi_files:
                #Add target filename to h5_keys array
                h5_keys.append(obj["Key"])

print(f"Found {len(h5_keys)} GEDI HDF5 files.")
if h5_keys:
    print(f"  First : {h5_keys[0]}")
    print(f"  Last  : {h5_keys[-1]}")

Scanning s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/ for GEDI HDF5 files...Only use county specific files.
Found 52 GEDI HDF5 files.
  First : GEDI/GEDI02_A/002/GEDI02_A_2019141235737_O02487_02_T00138_02_003_01_V002.h5
  Last  : GEDI/GEDI02_A/002/GEDI02_A_2025173142104_O36969_03_T05926_02_004_02_V002.h5


## Phase 1 & 1a: Batch Extraction with Spatial Masking and Quality Filtering

In [7]:
# Batch extraction helper function

def extract_gedi_dataframes(h5_keys, max_workers=16):
    """
    Concurrently download and process GEDI .h5 granules from S3, returning
    a combined list of per-beam DataFrames that passed spatial/quality filtering.

    Parameters
    ----------
    h5_keys : list[str]
        S3 object keys for the .h5 granules to process.
    max_workers : int, optional
        Number of concurrent threads to use for downloading/processing files
        (default: 16). Network I/O bound, so pushing this higher is usually
        safe until you hit S3 throttling or bandwidth limits.

    Returns
    -------
    list[pd.DataFrame]
        List of per-beam DataFrames, one per beam that had valid data after
        spatial masking and quality filtering.
    """
    all_dfs = []
    cell_start_time = time.time()
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_one_file, key): key for key in h5_keys}
        for future in as_completed(futures):
            key = futures[future]
            try:
                file_dfs = future.result()
                all_dfs.extend(file_dfs)
            except Exception as e:
                print(f"Unhandled error processing {os.path.basename(key)}: {e}")
            completed += 1
            if completed % 10 == 0:
                elapsed = time.time() - cell_start_time
                print(f"Processed {completed}/{len(h5_keys)} files... "
                      f"({elapsed:.2f}s elapsed total)")

    total_elapsed = time.time() - cell_start_time
    minutes, seconds = divmod(total_elapsed, 60)
    print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
    print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")

    return all_dfs

print("Batch extraction helper function defined.")

Batch extraction helper function defined.


In [8]:
fs = s3fs.S3FileSystem(anon=False)

all_dfs = extract_gedi_dataframes(h5_keys, max_workers=9)  #Max Workers can be set to 8,16 or 24 


Processed 10/52 files... (158.29s elapsed total)
Processed 20/52 files... (277.77s elapsed total)
Processed 30/52 files... (389.00s elapsed total)
Processed 40/52 files... (514.56s elapsed total)
Processed 50/52 files... (608.47s elapsed total)

Extraction complete. Beams with valid data: 217
🚀 Total execution time: 10m 10.65s


## Save Extracted Points to Parquet

In [9]:
if not all_dfs:
    raise RuntimeError("No valid GEDI shots found within the bounding box. Check S3 prefix and bbox settings.")

df_gedi_route29north = pd.concat(all_dfs, ignore_index=True)
df_gedi_route29north.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Total valid GEDI shots saved : {len(df_gedi_route29north):,}")
print(f"Years covered                : {sorted(df_gedi_route29north['year'].unique())}")
print(f"Parquet file                 : {OUTPUT_PARQUET}")
df_gedi_route29north.head()

Total valid GEDI shots saved : 15,996
Years covered                : [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Parquet file                 : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_multiyear-route29north.parquet


,longitude,latitude,quality_flag,sensitivity,rh98,year,file_source,beam
0,-78.402428,38.231998,1,0.970914,3.140000,2019,GEDI02_A_2019285212909_O04720_03_T00387_02_003...,BEAM0110
1,-78.401928,38.231669,1,0.970822,2.840000,2019,GEDI02_A_2019285212909_O04720_03_T00387_02_003...,BEAM0110
2,-78.401425,38.231339,1,0.972102,15.630000,2019,GEDI02_A_2019285212909_O04720_03_T00387_02_003...,BEAM0110
3,-78.400924,38.231009,1,0.975513,30.360001,2019,GEDI02_A_2019285212909_O04720_03_T00387_02_003...,BEAM0110
4,-78.400425,38.230681,1,0.974105,23.370001,2019,GEDI02_A_2019285212909_O04720_03_T00387_02_003...,BEAM0110


## Save Extracted data points to CSV locally to support Canopy Height Change statistics

In [10]:
#Save to CSV locally to support Canopy Height Change
df_gedi_route29north.to_csv(OUTPUT_DETAILED_CANOPY_HEIGHT_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_CANOPY_HEIGHT_CSV}")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_height-route29north.csv


## Phase 1b: Harmonize to the SMAP 9 km Grid

In [11]:
lon_bins = np.arange(MIN_LON, MAX_LON, GRID_RES)
lat_bins = np.arange(MIN_LAT, MAX_LAT, GRID_RES)

df_gedi_route29north['lon_grid'] = pd.cut(df_gedi_route29north['longitude'], bins=lon_bins, labels=lon_bins[:-1]).astype(float)
df_gedi_route29north['lat_grid'] = pd.cut(df_gedi_route29north['latitude'],  bins=lat_bins, labels=lat_bins[:-1]).astype(float)

gedi_grid_route29north = df_gedi_route29north.groupby(['year', 'lat_grid', 'lon_grid'])['rh98'].mean().reset_index()

ds_gedi = gedi_grid_route29north.set_index(['year', 'lat_grid', 'lon_grid']).to_xarray()
ds_gedi.to_netcdf(OUTPUT_NETCDF)

print(f"Grid cells produced for the City of Charlottesville : {len(gedi_grid_route29north):,}")
print(f"NetCDF saved to     : {OUTPUT_NETCDF}")
gedi_grid_route29north.head()

Grid cells produced for the City of Charlottesville : 0
NetCDF saved to     : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_grid-route29north.nc


,year,lat_grid,lon_grid,rh98


## Phase 1c: Fetch Jurisdiction Boundaries from US Census TIGERweb

In [13]:
boundary_gdfs = []

for name, st_fips, geo_id, geo_type in TARGET_JURISDICTIONS:
    print(f"Current Name: {name}")
    try:
        if name == "Albemarle": 
            b_gdf = fetch_boundary(name, st_fips, geo_id, geo_type)
            boundary_gdfs.append(b_gdf)
    except Exception as e:
        print(f"  Failed to fetch boundary for {name}: {e}")

if not boundary_gdfs:
    raise RuntimeError("No jurisdiction boundaries fetched. Cannot perform spatial join.")

filtered_counties = pd.concat(boundary_gdfs, ignore_index=True)
print(f"\nBoundaries fetched: {len(filtered_counties)} jurisdiction(s)")
filtered_counties[['jurisdiction', 'geometry']].head(10)

Current Name: Albemarle
  Fetching boundary for Albemarle...
Current Name: Augusta
Current Name: Buckingham
Current Name: Charlottesville
Current Name: Fluvanna
Current Name: Greene
Current Name: Louisa
Current Name: Nelson
Current Name: Rockingham

Boundaries fetched: 1 jurisdiction(s)


,jurisdiction,geometry
0,Albemarle,"POLYGON ((-78.64831 37.73272, -78.64824 37.732..."


## 10 — Phase 1c: Spatial Join and County-Level Aggregation

In [14]:
gdf_gedi_route29north = gpd.GeoDataFrame(
    df_gedi_route29north,
    geometry=gpd.points_from_xy(df_gedi_route29north.longitude, df_gedi_route29north.latitude),
    crs="EPSG:4326"
)

print("Performing spatial join...")
gedi_with_county_route29north = gpd.sjoin(
    gdf_gedi_route29north,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

gedi_county_summary_route29north = (
    gedi_with_county_route29north
    .groupby(['year', 'jurisdiction'])['rh98']
    .mean()
    .reset_index()
    .rename(columns={'rh98': 'canopy_height_mean_m'})
)

print(f"Spatial join complete. Rows in summary: {len(gedi_county_summary_route29north)}")
gedi_county_summary_route29north

Performing spatial join...
Spatial join complete. Rows in summary: 7


,year,jurisdiction,canopy_height_mean_m
0,2019,Albemarle,19.381147
1,2020,Albemarle,18.555693
2,2021,Albemarle,16.550257
3,2022,Albemarle,16.487835
4,2023,Albemarle,14.535197
5,2024,Albemarle,18.342386
6,2025,Albemarle,17.620771


In [15]:
# Load dataframe with canopy height data points
df_gedi_route29north_canopy_height = pd.read_csv(OUTPUT_DETAILED_CANOPY_HEIGHT_CSV)

gdf_gedi_route29north_canopy_height = gpd.GeoDataFrame(
     df_gedi_route29north_canopy_height,
     geometry=gpd.points_from_xy(df_gedi_route29north_canopy_height.longitude, df_gedi_route29north_canopy_height.latitude),
     crs="EPSG:4326"
 )

# The Fix for the Value Error

# Drop any stale index columns from a previous join
gdf_gedi_route29north_canopy_height = gdf_gedi_route29north_canopy_height.drop(
    columns=[c for c in ["index_right", "index_left"] if c in gdf_gedi_route29north_canopy_height.columns]
)
filtered_counties = filtered_counties.drop(
    columns=[c for c in ["index_right", "index_left"] if c in filtered_counties.columns]
)

print("Performing spatial join...")
gedi_with_county_canopy_height_route29north = gpd.sjoin(
    gdf_gedi_route29north_canopy_height,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

print(f"Spatial join complete. Rows in summary: {len(gedi_with_county_canopy_height_route29north)}")
gedi_with_county_canopy_height_route29north.head(20)

Performing spatial join...
Spatial join complete. Rows in summary: 12061


,longitude,latitude,quality_flag,sensitivity,rh98,year,file_source,beam,geometry,index_right,jurisdiction
166,-78.399338,38.182695,1,0.924650,21.57,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0000,POINT (-78.39934 38.18269),0,Albemarle
206,-78.464360,38.213595,1,0.965851,21.09,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0010,POINT (-78.46436 38.2136),0,Albemarle
207,-78.463858,38.213266,1,0.960628,27.98,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0010,POINT (-78.46386 38.21327),0,Albemarle
208,-78.463356,38.212936,1,0.969141,27.31,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0010,POINT (-78.46336 38.21294),0,Albemarle
209,-78.462854,38.212605,1,0.959031,23.67,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0010,POINT (-78.46285 38.21261),0,Albemarle
210,-78.462353,38.212274,1,0.924680,10.75,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0010,POINT (-78.46235 38.21227),0,Albemarle
211,-78.461851,38.211945,1,0.965667,21.54,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0010,POINT (-78.46185 38.21195),0,Albemarle
212,-78.460848,38.211286,1,0.948887,26.63,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0010,POINT (-78.46085 38.21129),0,Albemarle
213,-78.460346,38.210957,1,0.961040,3.78,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0010,POINT (-78.46035 38.21096),0,Albemarle
214,-78.459845,38.210627,1,0.945918,3.03,2019,GEDI02_A_2019181144908_O03102_03_T01810_02_003...,BEAM0010,POINT (-78.45984 38.21063),0,Albemarle


In [16]:
#Save to CSV to support County Canopy Height Change
gedi_with_county_canopy_height_route29north.to_csv(OUTPUT_COUNTY_CANOPY_HEIGHT_CSV, index=False)
print(f"Saved locally to : {OUTPUT_COUNTY_CANOPY_HEIGHT_CSV}")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_height-route29north.csv


## Save City of Charlottesville details Locally and Upload to S3

In [17]:
# Save locally
gedi_county_summary_route29north.to_csv(OUTPUT_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_COUNTY_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_COUNTY_CSV)
s3_county_key       = GEDI_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_COUNTY_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll summary phases completed successfully!")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_summary-route29north.csv
Uploaded to S3   : s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary-route29north.csv

All summary phases completed successfully!


In [18]:
# Save locally
gedi_with_county_canopy_height_route29north.to_csv(OUTPUT_DETAILED_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_COUNTY_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_DETAILED_COUNTY_CSV)
s3_county_key       = GEDI_DETAILED_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_DETAILED_COUNTY_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll detailed phases completed successfully!")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_detailed-route29north.csv
Uploaded to S3   : s3://central-virginia-tree-canopy-project/gedi-county-detailed/virginia_gedi_county_detailed-route29north.csv

All detailed phases completed successfully!
